# 03 hnsc_pool_freeze_final

Source code preserved from the supplied analysis notebook. Saved outputs and execution counts have been cleared for public distribution. See README for input requirements and limitations.


In [ ]:
import pandas as pd
import json
from collections import defaultdict

pool = pd.read_csv("full_candidate_pool_with_coverage.csv")
print("Starting candidate pool:", pool.shape[0], "genes")
print("Already resolved in TCGA-HNSC (direct + known alias):", pool['In_TCGA_HNSC'].sum())

In [ ]:
missing = pool[~pool['In_TCGA_HNSC']].copy()

def priority(row):
    if any(s in row['Sources'] for s in ['Toustrup', 'NIMRAD', 'BreastScore']):
        return 'HIGH (named signature)'
    if row['N_sources'] >= 3:
        return 'MEDIUM (multi-source)'
    return 'LOW (single general source)'

missing['Priority'] = missing.apply(priority, axis=1)
print(missing['Priority'].value_counts())
print("\nNo unresolved genes belong to a named HNSC signature -- good starting position.")

In [ ]:
# Candidate older/alternative symbols for the remaining unresolved genes,
# applied where a reasonably confident alias exists. Anything not resolved here
# is documented as genuinely unresolved, per Jimmy's "don't over-chase" guidance.

candidate_aliases = {
    "NOCT": ["CCRN4L"], "SIRPA": ["SHPS1", "PTPNS1", "MYD1", "BIT"],
    "DDR1": ["EDDR1", "NEP", "PTK3A", "CAK"], "CADM1": ["TSLC1", "SYNCAM1", "IGSF4"],
    "UTP11": ["UTP11L"], "FOSL2": ["FRA2"], "TEK": ["TIE2", "VMCM1"],
    "BCAN": ["BEHAB"], "EGLN1": ["PHD2", "SM20", "HIFPH2"],
    "SLC16A2": ["MCT8", "XPCT"], "SLC37A4": ["G6PT1"], "TUBA1C": ["TUBA6"],
    "GCNT2": ["NAGCT2", "BGAT2", "ULG2"], "HSPH1": ["HSP110"],
    "NREP": ["P311", "C5ORF13"], "SAMD4A": ["SMAUG1"],
    "MT-CO1": ["MTCO1"], "MT-CO2": ["MTCO2"], "MT-ND5": ["MTND5"],
    "RBPJ": ["RBPSUH", "RBPJK", "IGKJRB1", "KYOT2", "CBF1"],
}

cols = pd.read_csv("clinical_rna_fire_combine.csv", nrows=0).columns.tolist()
cols_set = set(cols)

resolved_this_pass = {}
for gene, aliases in candidate_aliases.items():
    found = next((a for a in aliases if a in cols_set), None)
    if found:
        resolved_this_pass[gene] = found

print("Resolved this pass:", resolved_this_pass)
print("\nStill genuinely unresolved after reasonable checking:",
      [g for g in candidate_aliases if g not in resolved_this_pass])

In [ ]:
# Apply resolutions (including the 6 found in the previous session) to the pool
all_alias_fixes = {
    "CCN1": "CYR61", "CCN2": "CTGF", "CCN5": "WISP2", "CAVIN1": "PTRF",
    "CAVIN3": "PRKCDBP", "ERO1A": "ERO1L",
}
all_alias_fixes.update(resolved_this_pass)

for gene in all_alias_fixes:
    pool.loc[pool['Gene'] == gene, 'In_TCGA_HNSC'] = True

n_resolved = pool['In_TCGA_HNSC'].sum()
print(f"TCGA-HNSC coverage after alias resolution: {n_resolved} / {pool.shape[0]} ({100*n_resolved/pool.shape[0]:.1f}%)")

still_unresolved = pool[~pool['In_TCGA_HNSC']]
print(f"\nGenuinely unresolved, documented: {still_unresolved.shape[0]} genes")
print(still_unresolved[['Gene', 'Sources']].to_string(index=False))

In [ ]:
probe_to_symbol = {}
with open("GPL10558_HumanHT-12_V4_0_R2_15002873_B.txt", "r", encoding="latin-1") as f:
    for line in f:
        if line.startswith("Species\tSource\tSearch_Key"):
            header = line.strip().split("\t")
            break
    probe_idx = header.index("Probe_Id")
    symbol_idx = header.index("Symbol")

    for line in f:
        if line.startswith("[Controls]") or line.strip() == "":
            break
        fields = line.rstrip("\n").split("\t")
        if len(fields) <= max(probe_idx, symbol_idx):
            continue
        probe_id, symbol = fields[probe_idx].strip(), fields[symbol_idx].strip()
        if probe_id and symbol:
            probe_to_symbol[probe_id] = symbol

print("Total probes with a valid Probe_Id + Symbol on this platform:", len(probe_to_symbol))

In [ ]:
import openpyxl

wb = openpyxl.load_workbook("GSE65858.xlsx", data_only=True, read_only=True)
ws = wb["Sheet1"]

# Row 106 onward is the actual expression data; column A holds the probe ID
gse_probe_ids = [str(row[0]) for row in ws.iter_rows(min_row=106, max_row=ws.max_row, max_col=1, values_only=True) if row[0]]
print("Probe rows actually present in the GSE65858 data matrix:", len(gse_probe_ids))

In [ ]:
symbol_to_probes = defaultdict(list)
for probe_id in gse_probe_ids:
    symbol = probe_to_symbol.get(probe_id)
    if symbol:
        symbol_to_probes[symbol].append(probe_id)

print("Unique gene symbols measurable in GSE65858:", len(symbol_to_probes))
multi_probe = {g: p for g, p in symbol_to_probes.items() if len(p) > 1}
print("Genes with more than one probe (collapsed via mean, per the fixed rule above):", len(multi_probe))

In [ ]:
# All alias resolutions found across both the TCGA-HNSC and GSE65858 checks
ALL_ALIASES = dict(all_alias_fixes)
ALL_ALIASES["MRGBP"] = "C20orf20"

def check_match(gene, direct_set, alias_lookup_fn):
    if gene in direct_set:
        return "Direct", ""
    if gene in ALL_ALIASES and alias_lookup_fn(ALL_ALIASES[gene]):
        return "Alias", ALL_ALIASES[gene]
    return "None", ""

documented_rows = []
for _, r in pool.iterrows():
    gene = r["Gene"]
    tcga_match, tcga_alias = check_match(gene, cols_set, lambda a: a in cols_set)
    gse_match, gse_alias = check_match(gene, symbol_to_probes, lambda a: a in symbol_to_probes)
    n_probes = len(symbol_to_probes.get(gene if gse_match == "Direct" else gse_alias, []))
    documented_rows.append({
        "Gene": gene, "N_sources": r["N_sources"], "Sources": r["Sources"],
        "TCGA_HNSC_Match": tcga_match, "TCGA_HNSC_Alias_Used": tcga_alias,
        "GSE65858_Match": gse_match, "GSE65858_Alias_Used": gse_alias,
        "GSE65858_N_Probes": n_probes,
    })

pool = pd.DataFrame(documented_rows)
pool["In_TCGA_HNSC"] = pool["TCGA_HNSC_Match"] != "None"
pool["In_GSE65858"] = pool["GSE65858_Match"] != "None"
pool["In_Both"] = pool["In_TCGA_HNSC"] & pool["In_GSE65858"]

n_gse = pool["In_GSE65858"].sum()
print(f"GSE65858 coverage: {n_gse} / {pool.shape[0]} ({100*n_gse/pool.shape[0]:.1f}%)")

n_both = pool["In_Both"].sum()
print(f"\n*** FROZEN POOL: measurable in BOTH cohorts: {n_both} / {pool.shape[0]} ({100*n_both/pool.shape[0]:.1f}%) ***")

pool.to_csv("full_candidate_pool_final_coverage.csv", index=False)

alias_resolved = pool[(pool["TCGA_HNSC_Match"]=="Alias") | (pool["GSE65858_Match"]=="Alias")]
print(f"\nGenes resolved via alias on at least one side ({alias_resolved.shape[0]} total):")
alias_resolved[["Gene","TCGA_HNSC_Match","TCGA_HNSC_Alias_Used","GSE65858_Match","GSE65858_Alias_Used"]]

In [ ]:
for sig in ['Toustrup', 'NIMRAD', 'BreastScore']:
    sub = pool[pool['Sources'].str.contains(sig)]
    n_ok = sub['In_Both'].sum()
    print(f"{sig}: {n_ok} / {sub.shape[0]} genes usable in both cohorts")
    missing_sig = sub[~sub['In_Both']]
    if not missing_sig.empty:
        print("  Not usable in both:", missing_sig['Gene'].tolist())

In [ ]:
frozen_pool = pool[pool['In_Both']].sort_values(['N_sources', 'Gene'], ascending=[False, True])
frozen_pool.to_csv("FROZEN_candidate_pool.csv", index=False)
print("Frozen candidate pool saved:", frozen_pool.shape[0], "genes")
frozen_pool.head(15)